In [3]:
import pandas as pd
import numpy as np

In [4]:
# Load the dataset
df = pd.read_csv(r'E:\DA\DA\Python - Cohort Analysis\Dataset\online_retail_II.csv')

In [5]:
# Display the first few rows of the dataset
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_rows', 1000)

<span style="font-size: 15px;">**1. Remove Duplicates**</span>

In [ ]:
df.drop_duplicates(inplace=True)
print(f"After removing duplicates: {len(df)} rows")

<span style="font-size: 15px;">**2. Standardize Column Names & Data Types**</span>

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

df['description'] = df['description'].astype(str).str.strip().str.capitalize()
df['customer_id'] = pd.to_numeric(df['customer_id'], errors='coerce').astype('Int64')
df['customer_id'] = df['customer_id'].astype(str).str.strip()
df['country'] = (df['country'].astype(str).str.strip()
                 .str.replace('USA', 'United States')
                 .str.replace('EIRE', 'Ireland')
                 .str.replace('RSA', 'South Africa'))
df['invoicedate'] = pd.to_datetime(df['invoicedate'], errors='coerce')


<span style="font-size: 15px;">**3. Handle Missing Values**</span>

In [ ]:
print("Missing values before handling:")
print(df.isnull().sum())

df['customer_id'] = df['customer_id'].replace('<NA>', 'Guests')
df['description'] = df['description'].replace('<NA>', 'Unknown')


<span style="font-size: 15px;">**4. Logical Consistency Checks**</span>

In [ ]:
print(f"Starting rows: {len(df)}")

# Remove unspecified country
df = df[df['country'] != 'Unspecified']

# Remove zero or negative prices
df = df[df['price'] > 0]


# Validate stock code format (5 digits + optional letter)
pattern = r'^\d{5}[a-zA-Z]?$'
df = df[df['stockcode'].astype(str).str.match(pattern, na=False)]


# Validate cancellation logic
condition_cancel1 = (df['invoice'].astype(str).str.startswith('C')) & (df['quantity'] < 0)
condition_cancel2 = (~df['invoice'].astype(str).str.startswith('C')) & (df['quantity'] > 0)
df = df[condition_cancel1 | condition_cancel2]


<span style="font-size: 15px;">**5. Extract Product ID & Suffix**</span>

In [ ]:
extract_pattern = r'(\d{5})([a-zA-Z]?)'
df[['product_id', 'suffix']] = df['stockcode'].astype(str).str.extract(extract_pattern, expand=True)
df['suffix'] = df['suffix'].fillna('').replace('', 'None')


<span style="font-size: 15px;">**6. Calculate Total Sales**</span>

In [ ]:
df['total_sales'] = df['quantity'] * df['price']


<span style="font-size: 15px;">**7. Reorder Columns & Display Final Data**</span>

In [ ]:
columns_order = ['invoice', 'invoicedate', 'product_id', 'suffix', 
                 'description', 'customer_id', 'country',
                 'quantity', 'price', 'total_sales']

df = df[columns_order]
df.info()

<span style="font-size: 15px;">**8. Save Cleaned Dataset**</span>

In [ ]:
output_path = r'E:\DA\DA\Python - Cohort Analysis\Dataset\online_retail_II_cleaned.csv'
df.to_csv(output_path, index=False)
